In [1]:
# R5 ROSALIA pointing
# Alejandro Borlaff, NASA ARC, 2026-04-02 
# Implementing the method by Mario Gennaro to estimate the best position angle + tolerance angle for Roman Space Telescope given a date. 
import os
import numpy as np
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
import rosalia as rs

In [ ]:
    def find_wfi_center_for_offset_target(ra_target, dec_target, mjd, dX, dY, PA_wfi=None, verbose=False):
        # dX dY in degrees
    
        import pysiaf
        # Define the Roman Space Telescope Frame 
        rsiaf = pysiaf.Siaf('Roman')
        wfi_cen = rsiaf['WFI_CEN']

        # Find the optimal position angle of the observatory at that RA, Dec, and date
        # if PA_wfi is None:
        # This is the position angle of the observatory
        # WFI focal plane Y direction is -60 degrees offset from this. 
        PA_v3 = rs.telescopes.Roman.get_bestPA(ra=ra_target, dec=dec_target, mjd=mjd)
            #PA_wfi = PA_v3 - 60*u.degree
        # else:
            #PA_v3 = PA_wfi + 60*u.degree
        
        if verbose:
            print("RA: " + str(ra_target) + " - DEC: " + str(dec_target) + " PA_v3: " + str(PA_v3))
    
        # First, we find v2, v3 for the stray light point we want to hit
        # these are the coordinates in the telescope (V2, V3) plane, 
        # with origin in WFI_CEN, the center of the focal plane array.
        v2, v3 = wfi_cen.idl_to_tel(dX*60*60, dY*60*60, method="spherical", 
                                    input_coordinates="polar", output_coordinates="polar")


        # Then we define the attitude matrix required to place that v2 and v3 on the star
        attmat = pysiaf.utils.rotations.attitude_matrix(nu2=v2, nu3=v3,
                                                    ra=ra_target, dec=dec_target, 
                                                    pa=PA_v3)

        # We apply the attitude matrix to the observatory
        wfi_cen.set_attitude_matrix(attmat)
        # compute the position angle at that aperture
        V2Ref = rsiaf.apertures['WFI_CEN'].V2Ref
        V3Ref = rsiaf.apertures['WFI_CEN'].V3Ref
        pa = pysiaf.utils.rotations.posangle(attmat, V2Ref, V3Ref)
    
        # Compute the sky coordinates of the WFI_CEN aperture reference position
        wfi_ra, wfi_dec = wfi_cen.idl_to_sky(0, 0)
        if verbose: print(f'' + name.iloc[i] + f' - WFI_CEN: RA = {wfi_ra:.5f} deg, Dec = {wfi_dec:.5f} deg')
        return({"ra_wficen": wfi_ra, "dec_wficen": wfi_dec, "pa_wfi": pa - 60, "V3PA": pa, "PA_v3": PA_v3})

In [36]:
# Let's find Canopus (alpha Carinae), the star selected for CAR171. 
# This star is in the continuous viewing zone of Roman, and it is the second brightest star in the sky.
import astropy.units as u
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord

alfCar = Simbad.query_object('Canopus')
ra_star = alfCar["ra"][0]
dec_star = alfCar["dec"][0]
epoch = Time('2026-12-01T00:00:00', scale='utc')
dX = -3.7000 	
dY =  5.2700

print(alfCar["ra"])
print(alfCar["dec"])

PAV3i = rs.telescopes.Roman.get_bestPA(ra_star, dec_star, epoch.mjd)
print(PAV3i)
offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=ra_star,
                                                      dec_target=dec_star,
                                                      mjd=epoch.mjd,
                                                      dX=dX, dY=dY)

offset_pointing

        ra       
       deg       
-----------------
95.98795782918306
       dec        
       deg        
------------------
-52.69566138386201
117.644035


{'ra_wficen': 85.51955135055718,
 'dec_wficen': -51.94857615946075,
 'pa_wfi': 65.81089345663555,
 'V3PA': 125.81089345663555,
 'PA_v3': 117.644035}

In [3]:
print(PAV3i)

117.644035


In [37]:

epoch = Time('2026-12-01T00:00:00', scale='utc')

target = [SkyCoord(96.290898*u.deg, -52.657578*u.deg, frame="icrs")]
print(target[0])

c = rs.point.compute_visibility(target,report=True,fileout='test.txt',interval_sampling_days=None,interval_start_time=epoch,interval_duration_days=1)
c.compute_and_display()
np.float32(c.df_results["pa_obs_y"])

<SkyCoord (ICRS): (ra, dec) in deg
    (96.290898, -52.657578)>


array([117.92104], dtype=float32)

In [38]:
coords = SkyCoord(96.173075*u.deg, -52.545470*u.deg, frame="icrs")
print(coords.ra.hms)
print(coords.dec.dms)

hms_tuple(h=6.0, m=24.0, s=41.53800000000402)
dms_tuple(d=-52.0, m=-32.0, s=-43.69200000000603)


In [39]:
c

In [ ]:
c.compute_and_display()

In [44]:
db = c.df_results
db


,,Sun_RA,Sun_Dec,separation,good_angles,nominal_roll,pa_obs_y,pa_fpa_local_x,pa_fpa_local_y,sunang_x,sunang_y,sunang_z
"(RA, Dec)",DOY,,,,,,,,,,,
"(6h25m09.81552s, -52d39m27.2808s)",2026-335.00000,246.67264,-21.704704,101.298021,True,207.921041,117.921041,177.921041,87.921041,101.298242,90.0,11.298242


In [45]:
db["pa_obs_y"]

(RA, Dec)                          DOY           
(6h25m09.81552s, -52d39m27.2808s)  2026-335.00000    117.921041
Name: pa_obs_y, dtype: object

In [48]:
db["pa_fpa_local_y"] + 30

(RA, Dec)                          DOY           
(6h25m09.81552s, -52d39m27.2808s)  2026-335.00000    117.921041
Name: pa_fpa_local_y, dtype: object

In [13]:

ra_target = ra_star
dec_target = dec_star
mjd = epoch.mjd
verbose = True

if True:
    if True:
        import pysiaf
        # Define the Roman Space Telescope Frame 
        rsiaf = pysiaf.Siaf('Roman')
        wfi_cen = rsiaf['WFI_CEN']

        # Find the optimal position angle of the observatory at that RA, Dec, and date
        # if PA_wfi is None:
        # This is the position angle of the observatory
        # WFI focal plane Y direction is -60 degrees offset from this. 
        PA_v3 = rs.telescopes.Roman.get_bestPA(ra=ra_target, dec=dec_target, mjd=mjd)
            #PA_wfi = PA_v3 - 60*u.degree
        # else:
            #PA_v3 = PA_wfi + 60*u.degree
        

        # First, we find v2, v3 for the stray light point we want to hit
        # these are the coordinates in the telescope (V2, V3) plane, 
        # with origin in WFI_CEN, the center of the focal plane array.
        v2, v3 = wfi_cen.idl_to_tel(dX*60*60, dY*60*60, method="spherical", 
                                    input_coordinates="polar", output_coordinates="polar")


        # Then we define the attitude matrix required to place that v2 and v3 on the star
        attmat = pysiaf.utils.rotations.attitude_matrix(nu2=v2, nu3=v3,
                                                    ra=ra_target, dec=dec_target, 
                                                    pa=PA_v3)

        # We apply the attitude matrix to the observatory
        wfi_cen.set_attitude_matrix(attmat)
        # compute the position angle at that aperture
        V2Ref = rsiaf.apertures['WFI_CEN'].V2Ref
        V3Ref = rsiaf.apertures['WFI_CEN'].V3Ref
        pa = pysiaf.utils.rotations.posangle(attmat, V2Ref, V3Ref)
        if verbose:
            print("RA: " + str(ra_target) + " - DEC: " + str(dec_target) + " PA_v3: " + str(pa))
    

RA: 95.98795782918306 - DEC: -52.69566138386201 PA_v3: 125.81089345663555


In [25]:
if True:
    if True: 
        # Then we define the attitude matrix required to place that v2 and v3 on the star
        attmat = pysiaf.utils.rotations.attitude_matrix(nu2=v2, nu3=v3,
                                                    ra=ra_target, dec=dec_target, 
                                                    pa=pa)

        # We apply the attitude matrix to the observatory
        wfi_cen.set_attitude_matrix(attmat)
        # compute the position angle at that aperture
        V2Ref = rsiaf.apertures['WFI_CEN'].V2Ref
        V3Ref = rsiaf.apertures['WFI_CEN'].V3Ref
        pa = pysiaf.utils.rotations.posangle(attmat, V2Ref, V3Ref)
        if verbose:
            print("RA: " + str(ra_target) + " - DEC: " + str(dec_target) + " PA_v3: " + str(pa))

RA: 95.98795782918306 - DEC: -52.69566138386201 PA_v3: -174.05971670840609


In [41]:
360/12

30.0